# Model Training using the TensorFlow Object Detection API

## Objective

The objective of this notebook is to train and export a deep learning object detection model using the TensorFlow Object Detection API based on the pre-processed and validated dataset.

This notebook covers:
- Annotation sanitisation
- Dataset partitioning
- TFRecord generation
- Model selection and configuration
- Model training and export for inference


## Annotation Sanitisation

Prior to training, XML annotation files were sanitised to remove trailing whitespace and formatting inconsistencies.

Even minor annotation errors can result in:
- Class mismatches during training
- Silent exclusion of bounding boxes
- Training instability or script failure

This step ensures annotation consistency across both training and testing datasets.


In [1]:
import glob
import xml.etree.ElementTree as ET

xml_dir = "./annotated_images/train"

for xml_file in glob.glob(f"{xml_dir}/*.xml"):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    # Fix filename
    fn = root.find("filename")
    if fn is not None and fn.text:
        fn.text = fn.text.strip()

    # Fix object names
    for obj in root.findall("object"):
        name = obj.find("name")
        if name is not None and name.text:
            name.text = name.text.strip()

    tree.write(xml_file, encoding="utf-8")

print("XML cleanup done")


XML cleanup done


In [1]:
!python xmlconversion.py --verbose

images\OryxGazella_CDB_S1_A05_R2_IMAG0334.xml
images\OryxGazella_CDB_S1_A05_R2_IMAG0868.xml
images\OryxGazella_CDB_S1_B06_R1_IMAG0127.xml
images\OryxGazella_CDB_S1_B06_R1_IMAG0136.xml
images\OryxGazella_CDB_S1_B06_R1_IMAG0139.xml
images\OryxGazella_CDB_S1_B06_R1_IMAG0140.xml
images\OryxGazella_CDB_S1_B06_R1_IMAG0144.xml
images\OryxGazella_CDB_S1_B06_R1_IMAG0145.xml
images\OryxGazella_CDB_S1_B06_R1_IMAG0204.xml
images\OryxGazella_CDB_S1_B06_R1_IMAG0217.xml
images\OryxGazella_CDB_S1_B06_R1_IMAG0218.xml
images\OryxGazella_CDB_S1_B06_R1_IMAG0219.xml
images\OryxGazella_CDB_S1_B06_R1_IMAG0225.xml
images\OryxGazella_CDB_S1_B06_R1_IMAG0226.xml
images\OryxGazella_CDB_S1_B06_R1_IMAG0228.xml
images\OryxGazella_CDB_S1_B06_R1_IMAG0256.xml
images\OryxGazella_CDB_S1_B06_R1_IMAG0257.xml
images\OryxGazella_CDB_S1_B06_R1_IMAG0258.xml
images\OryxGazella_CDB_S1_B06_R1_IMAG0259.xml
images\OryxGazella_CDB_S1_B06_R1_IMAG0260.xml
images\OryxGazella_CDB_S1_B06_R1_IMAG0261.xml
images\OryxGazella_CDB_S1_B06_R1_I

'rsync' is not recognized as an internal or external command,
operable program or batch file.


## Dataset Partitioning Strategy

A 90/10 train–test split was selected to maximise the number of samples available for training while retaining a sufficient subset for evaluation.

Given the modest size of the annotated dataset (1,500 images) and the complexity of object detection models, prioritising training data helps improve feature learning and localisation accuracy.


### Partition train/test split 90/10

In [15]:
!python partition_dataset.py -x -i ./annotated_images -r 0.1

In [3]:
import os
from glob import glob

xml_dir = "./images"
xml_files = [os.path.splitext(os.path.basename(f))[0] for f in glob(os.path.join(xml_dir, "*.xml"))]
jpg_files = [os.path.splitext(os.path.basename(f))[0] for f in glob(os.path.join(xml_dir, "*.jpg"))]

missing_images = [f for f in xml_files if f not in jpg_files]

print("Missing images for XMLs:", missing_images)


Missing images for XMLs: []


Annotation consistency checks confirmed that every XML annotation file has a corresponding image file, with no missing or mismatched samples identified.

This ensures that all annotated data will be correctly included during TFRecord generation and model training.


## TFRecord Generation

The TensorFlow Object Detection API requires datasets to be provided in the TFRecord format for efficient streaming and preprocessing.

TFRecords:
- Reduce I/O overhead
- Support large-scale datasets
- Enable consistent parsing during training and evaluation


### Update the .PBTXT File

In [18]:
!code ./data/label_map.pbtxt

### Create TF Training Record

In [29]:
!python generate_tfrecord.py \
  -x ./annotated_images/train \
  -l ./data/label_map.pbtxt \
  -o ./data/train.record \
  -i ./annotated_images/train


Successfully created the TFRecord file: ./data/train.record


### Create TF Testing Record

In [32]:
!python generate_tfrecord.py \
  -x ./annotated_images/test \
  -l ./data/label_map.pbtxt \
  -o ./data/test.record \
  -i ./annotated_images/test


Successfully created the TFRecord file: ./data/test.record


## Model Architecture Selection

The Faster R-CNN ResNet-101 architecture was selected from the TensorFlow Model Zoo for this experiment.

Faster R-CNN is a two-stage detector that prioritises detection accuracy and bounding box localisation over inference speed. This makes it particularly suitable for wildlife detection tasks, where precise localisation is more important than real-time performance.

The ResNet-101 backbone provides deep feature representations capable of capturing complex visual patterns, which is advantageous given the variability in animal appearance, scale, and background.


### Set Model Path

In [1]:
PATH_TO_MODEL = "./training/TF2/training/faster_rcnn_resnet101_v1_1024x1024_coco17_tpu-8"
PATH_TO_PIPELINE = PATH_TO_MODEL + "/pipeline.config"


In [2]:
!code {PATH_TO_PIPELINE}

## Hyperparameter Configuration

The number of training steps was set to 6,000 based on hardware constraints and expected convergence behaviour.

Longer training allows the model to refine both classification and localisation performance; however, excessive training risks overfitting and unnecessary computational cost. Convergence was later assessed in TensorBoard using the training loss curves and mAP at IoU thresholds of 0.50 and 0.75, evaluated at the final checkpoint.

## Model Training Execution

The TensorFlow training script was executed using the configured pipeline file. Training progress, including loss and evaluation metrics, was logged for later analysis using TensorBoard.



In [33]:
!python model_main_tf2.py \
  --model_dir=training/TF2/training/faster_rcnn_resnet101_v1_1024x1024_coco17_tpu-8 \
  --pipeline_config_path=training/TF2/training/faster_rcnn_resnet101_v1_1024x1024_coco17_tpu-8/pipeline.config \
  --num_train_steps=6000 \
  --alsologtostderr


^C


## Model Export for Inference

After training, the model was exported to a SavedModel format suitable for inference.

This step freezes the trained weights and graph structure, enabling deployment and evaluation on unseen test images in subsequent notebooks.


In [34]:
!python exporter_main_v2.py \
  --input_type image_tensor \
  --pipeline_config_path training/TF2/training/faster_rcnn_resnet101_v1_1024x1024_coco17_tpu-8/pipeline.config \
  --trained_checkpoint_dir training/TF2/training/faster_rcnn_resnet101_v1_1024x1024_coco17_tpu-8 \
  --output_directory training/TF2/training/faster_rcnn_resnet101_v1_1024x1024_coco17_tpu-8/saved_model


^C


## Discussion

This notebook demonstrates the complete training pipeline for a deep learning object detection model using the TensorFlow Object Detection API.

The chosen architecture prioritises localisation accuracy, aligning with the requirements of wildlife detection tasks. Model performance and convergence behaviour are evaluated in subsequent notebooks using TensorBoard and IoU-based metrics.


This notebook focuses on the Faster R-CNN ResNet-101 architecture, which is the only model trained and evaluated in these notebooks.
